# Text-Prompted Image Segmentation: Grounding DINO + SAM 2

**Series: Applied LLMs for Scientists**
**DigitalSreeni | github.com/bnsreenu**

---

In Notebook 1 we used Grounding DINO to detect objects using text prompts, producing bounding boxes.
Bounding boxes tell us *where* an object is, but not *which pixels* belong to it.

In this notebook we complete the pipeline: Grounding DINO provides the boxes, and SAM 2
(Segment Anything Model 2) converts each box into a precise pixel-level mask.
The result is a fully automated annotation pipeline driven by plain English descriptions.

**What we build:**

```
Text prompt -> Grounding DINO -> bounding boxes -> SAM 2 -> pixel masks
```

We test the same three images from Notebook 1:
- A natural COCO image (cats and remote controls) -- where the pipeline works well
- A kidney H&E pathology image (glomerulus) -- partial domain shift
- A Lucchi EM image (mitochondria) -- strong domain shift, expected failure

The final section compares boxes-only (Notebook 1) against masks (this notebook) side by side.

**Runtime:** GPU required. Go to `Runtime > Change runtime type > T4 GPU`.


---
## Section 1: Install and Import

Both models load via Hugging Face `transformers`, same as Notebook 1.
If you are running this immediately after Notebook 1 in the same Colab session,
the models may already be cached and loading will be fast.


In [ ]:
!pip install -q transformers accelerate tifffile gdown

In [ ]:
import transformers
from packaging import version
print(f"transformers version: {transformers.__version__}")
if version.parse(transformers.__version__) < version.parse("4.51.0"):
    print("Upgrading transformers...")
    import subprocess
    subprocess.run(["pip", "install", "-q", "-U", "transformers"], check=True)
    print("Done. Restart the runtime, then re-run from the top.")
else:
    print("Version OK.")


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import requests
import tifffile
import gdown
import io
from PIL import Image
from transformers import (
    AutoProcessor,
    AutoModelForZeroShotObjectDetection,
    Sam2Processor,
    Sam2Model,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cpu":
    print("Warning: running on CPU. Both models will be slow. Switch to T4 GPU.")


---
## Section 2: Load Both Models

We load Grounding DINO and SAM 2 once and reuse them throughout.

**Model choices:**
- Grounding DINO: `IDEA-Research/grounding-dino-base` -- same as Notebook 1
- SAM 2: `facebook/sam2.1-hiera-small` -- good balance of speed and mask quality on a T4 GPU.
  The `small` variant downloads ~180 MB versus ~900 MB for `large`,
  and quality difference is minor for single-object segmentation.

Both models download once and are then cached.


In [ ]:
GDINO_ID = "IDEA-Research/grounding-dino-base"
print("Loading Grounding DINO...")
gdino_processor = AutoProcessor.from_pretrained(GDINO_ID)
gdino_model = AutoModelForZeroShotObjectDetection.from_pretrained(GDINO_ID).to(device)
gdino_model.eval()
print("Grounding DINO ready.")


In [ ]:
SAM2_ID = "facebook/sam2.1-hiera-small"
print("Loading SAM 2 (~180 MB on first run)...")
sam2_processor = Sam2Processor.from_pretrained(SAM2_ID)
sam2_model = Sam2Model.from_pretrained(SAM2_ID).to(device)
sam2_model.eval()
print("SAM 2 ready.")


---
## Section 3: Helper Functions

We carry over `detect()`, `apply_nms()`, and `filter_large_boxes()` from Notebook 1 unchanged.
Two new functions are added:

- `segment_with_sam2()`: takes an image and bounding boxes from Grounding DINO, runs SAM 2
  on each box, and returns the best mask per box selected by highest IoU score
- `show_masks()`: overlays coloured semi-transparent masks on the image with labels
- `run_pipeline()`: convenience wrapper that chains the full detect -> filter -> segment flow

**SAM 2 box input format:**
The processor expects `input_boxes` as a list with shape `(1, N, 4)` when processing one image.
Formatting is handled inside `segment_with_sam2()`.

**Multiple mask candidates:**
SAM 2 returns 3 mask candidates per box by default. Each has an IoU score estimating its quality.
We always pick the highest-scoring candidate. This is handled inside `segment_with_sam2()`.


In [ ]:
from torchvision.ops import nms


# Grounding DINO helpers (carried over from Notebook 1)

def detect(image, text_prompts, box_threshold=0.35, text_threshold=0.25):
    """
    Run Grounding DINO on a PIL image.

    Parameters
    ----------
    image : PIL.Image (RGB)
    text_prompts : list of str
        Object descriptions without periods -- formatting handled internally.
    box_threshold : float
    text_threshold : float

    Returns
    -------
    dict with keys: boxes, scores, text_labels
    """
    cleaned = [p.lower().strip().rstrip('.') for p in text_prompts]
    text_string = ". ".join(cleaned) + "."

    inputs = gdino_processor(
        images=image,
        text=text_string,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = gdino_model(**inputs)

    results = gdino_processor.post_process_grounded_object_detection(
        outputs,
        inputs.input_ids,
        threshold=box_threshold,
        text_threshold=text_threshold,
        target_sizes=[image.size[::-1]]
    )
    return results[0]


def filter_large_boxes(results, image, max_area_fraction=0.7):
    """Remove boxes covering more than max_area_fraction of the image."""
    img_w, img_h = image.size
    img_area = img_w * img_h
    boxes  = results["boxes"]
    scores = results["scores"]
    labels = results.get("text_labels", results.get("labels", []))

    keep = []
    for i, box in enumerate(boxes):
        x0, y0, x1, y1 = box.tolist()
        if ((x1 - x0) * (y1 - y0)) / img_area < max_area_fraction:
            keep.append(i)

    return {
        "boxes":       boxes[keep],
        "scores":      scores[keep],
        "text_labels": [labels[i] for i in keep]
    }


def apply_nms(results, iou_threshold=0.5):
    """Remove overlapping duplicate boxes with Non-Maximum Suppression."""
    boxes  = results["boxes"]
    scores = results["scores"]
    labels = results.get("text_labels", results.get("labels", []))

    if len(boxes) == 0:
        return results

    keep = nms(boxes, scores, iou_threshold)
    return {
        "boxes":       boxes[keep],
        "scores":      scores[keep],
        "text_labels": [labels[i] for i in keep.tolist()]
    }


# SAM 2 helpers
def segment_with_sam2(image, boxes_tensor):
    """
    Run SAM 2 on a PIL image using bounding boxes from Grounding DINO.

    Parameters
    ----------
    image : PIL.Image (RGB)
    boxes_tensor : torch.Tensor  shape (N, 4)  xyxy pixel coordinates

    Returns
    -------
    masks : np.ndarray  shape (N, H, W)  boolean
    iou_scores : list of float  predicted IoU per mask
    """
    if len(boxes_tensor) == 0:
        return np.zeros((0, image.height, image.width), dtype=bool), []

    # SAM 2 processor expects input_boxes as [[[x0,y0,x1,y1], ...]]
    # shape: (1 image, N boxes, 4 coords)
    boxes_list = [boxes_tensor.cpu().numpy().tolist()]

    inputs = sam2_processor(
        images=image,
        input_boxes=boxes_list,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        # multimask_output=False: one mask per box instead of 3 candidates
        # simplifies output handling and is faster
        outputs = sam2_model(**inputs, multimask_output=False)

    # post_process_masks takes only original_sizes in the SAM 2 API
    # returns list (one per image) of boolean tensors shape (N, 1, H, W)
    masks_tensor = sam2_processor.post_process_masks(
        outputs.pred_masks.cpu(),
        inputs["original_sizes"]
    )[0]  # (N, 1, H, W)

    # iou_scores shape: (1, N, 1) or (N, 1) -- squeeze to get one score per box
    iou_raw = outputs.iou_scores.cpu().squeeze()  # -> (N,) or scalar if N=1
    if iou_raw.dim() == 0:
        iou_raw = iou_raw.unsqueeze(0)  # handle single-box edge case -> (1,)

    # masks_tensor shape (N, 1, H, W) -- squeeze the candidate dimension
    masks_np  = masks_tensor[:, 0, :, :].numpy()  # (N, H, W) boolean
    iou_list  = iou_raw.tolist()
    if not isinstance(iou_list, list):
        iou_list = [iou_list]

    return masks_np, iou_list

def show_masks(image, detection_results, masks, iou_scores,
               title="", alpha=0.45, figsize=(5, 4)):
    """
    Overlay SAM 2 masks on the image with coloured fills, box outlines, and labels.

    Parameters
    ----------
    image : PIL.Image
    detection_results : dict  output of detect() after NMS and large-box filtering
    masks : np.ndarray  (N, H, W)  boolean
    iou_scores : list of float
    title : str
    alpha : float  mask transparency
    figsize : tuple
    """
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    ax.imshow(image)
    ax.set_title(title, fontsize=13, pad=10)
    ax.axis("off")

    boxes  = detection_results["boxes"].cpu().numpy()
    scores = detection_results["scores"].cpu().numpy()
    labels = detection_results.get("text_labels", detection_results.get("labels", []))
    colors = plt.cm.tab10.colors

    for i, (box, det_score, label) in enumerate(zip(boxes, scores, labels)):
        color = colors[i % len(colors)]

        if i < len(masks):
            mask = masks[i]
            colored = np.zeros((*mask.shape, 4), dtype=float)
            colored[mask] = [*color[:3], alpha]
            ax.imshow(colored)

        x0, y0, x1, y1 = box
        rect = patches.Rectangle(
            (x0, y0), x1 - x0, y1 - y0,
            linewidth=2, edgecolor=color, facecolor="none"
        )
        ax.add_patch(rect)

        iou = iou_scores[i] if i < len(iou_scores) else 0.0
        ax.text(
            x0, y0 - 6,
            f"{label} det={det_score:.2f} iou={iou:.2f}",
            color="white", fontsize=8,
            bbox=dict(facecolor=color, alpha=0.85, pad=2, edgecolor="none")
        )

    plt.tight_layout()
    plt.show()
    print(f"Objects segmented: {len(boxes)}")
    for label, score, iou in zip(labels, scores, iou_scores):
        print(f"  {label}: detection={score:.3f}  SAM IoU={iou:.3f}")


def run_pipeline(image, prompts, box_threshold=0.25, title=""):
    """
    Run the full Grounding DINO -> SAM 2 pipeline on a PIL image.

    Returns
    -------
    results : dict   detection results after NMS and filtering
    masks   : np.ndarray  (N, H, W)
    iou_scores : list of float
    """
    results = detect(image, prompts, box_threshold=box_threshold)
    results = filter_large_boxes(results, image)
    results = apply_nms(results)

    if len(results["boxes"]) == 0:
        print("No detections above threshold. Try lowering box_threshold.")
        return results, np.array([]), []

    masks, iou_scores = segment_with_sam2(image, results["boxes"])
    show_masks(image, results, masks, iou_scores, title=title)
    return results, masks, iou_scores


---
## Section 4: COCO Natural Image -- Cats and Remote Controls

Same image as Notebook 1. Here we expect both detection and segmentation to work well.
The comparison in Section 7 will show how much more precise the masks are versus the boxes alone.


In [ ]:
coco_url = "http://images.cocodataset.org/val2017/000000039769.jpg"
response = requests.get(coco_url, timeout=15)
coco_image = Image.open(io.BytesIO(response.content)).convert("RGB")
print(f"Image size: {coco_image.size}")


In [ ]:
results_coco, masks_coco, iou_coco = run_pipeline(
    coco_image,
    prompts=["a cat", "a remote control"],
    box_threshold=0.25,
    title="COCO: Grounding DINO + SAM 2"
)


### What to observe

Each detected object now has a pixel-precise mask rather than a rectangular box. Notice:
- The cat masks follow the animal body outline, not a rectangle
- The remote control masks are tight to the object even where it overlaps with a cat

The label shows two scores: `det=` is Grounding DINO's detection confidence,
`iou=` is SAM 2's own estimate of how well its mask fits the object.
A high IoU score (above 0.85) generally means SAM 2 is confident the mask is accurate.


---
## Section 5: Kidney Pathology -- Glomerulus (H&E)

Same kidney image as Notebook 1. In Notebook 1 we got a correct bounding box with the
simple prompt "glomerulus". Now we take that box and ask SAM 2 to find the exact pixel boundary.

The glomerulus has a clear round boundary visible in H&E, so SAM 2 should be able to
delineate it even if Grounding DINO's box was slightly loose.


In [ ]:
kidney_url = "https://www.tau.ac.il/medicine/tau-only/webpath/jpeg1/renal129.jpg"
response = requests.get(kidney_url, timeout=15, headers={"User-Agent": "Mozilla/5.0"})
kidney_image = Image.open(io.BytesIO(response.content)).convert("RGB")
print(f"Image size: {kidney_image.size}")

plt.figure(figsize=(5, 4))
plt.imshow(kidney_image)
plt.title("Kidney H&E -- input image", fontsize=13)
plt.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
results_kidney, masks_kidney, iou_kidney = run_pipeline(
    kidney_image,
    prompts=["glomerulus"],
    box_threshold=0.25,
    title="Kidney: Grounding DINO + SAM 2"
)


### What to observe

Compare the mask boundary with the glomerulus edge visible in the H&E image.

**If the mask follows the glomerular capsule tightly:** SAM 2's image encoder picks up
the clear boundary between the glomerulus and surrounding tubular tissue, even without
histology training. SAM 2 responds to strong visual edges regardless of domain.

**If the mask is imprecise:** The domain gap still affects SAM 2's image encoder.
The mask may capture the rough region but miss fine structure like the Bowman's capsule.

Either way, the mask is more informative than the box and can serve as a starting point
for annotation that a human then refines -- which is exactly how these tools are used in practice.


---
## Section 6: Electron Microscopy -- Lucchi Dataset (Mitochondria)

Same EM stack as Notebook 1. In Notebook 1, Grounding DINO either failed or produced
unreliable boxes at low threshold. If there are no reliable boxes, the pipeline breaks
at the first stage -- SAM 2 has nothing to work with.

We run both scenarios here: standard threshold (likely no detections) and forced
low threshold (unreliable boxes fed to SAM 2) to show exactly what happens.


In [ ]:
gdrive_url = "https://drive.google.com/uc?export=download&id=1L7M-kAhpB1qM5bSpeCwrytboIEYuBjVX"
output_path = "lucchi_mini.tif"
gdown.download(gdrive_url, output_path, quiet=False)

stack = tifffile.imread(output_path)
if stack.ndim == 2:
    stack = stack[np.newaxis, ...]
print(f"Stack shape: {stack.shape}")

SLICE_IDX = 1
em_slice = stack[SLICE_IDX]
em_rgb   = Image.fromarray(em_slice).convert("RGB")

plt.figure(figsize=(5, 5))
plt.imshow(em_slice, cmap="gray")
plt.title(f"Lucchi EM -- slice {SLICE_IDX}", fontsize=13)
plt.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# Attempt 1: standard threshold -- likely produces no detections
print("Attempt 1: box_threshold=0.25")
results_em_1, masks_em_1, iou_em_1 = run_pipeline(
    em_rgb,
    prompts=["mitochondria"],
    box_threshold=0.25,
    title="EM: 'mitochondria' at threshold=0.25"
)


In [ ]:
# Attempt 2: very low threshold -- forces some detections to show what SAM 2 does with bad boxes
print("Attempt 2: box_threshold=0.05 (forcing detections)")
results_em_2 = detect(em_rgb, ["mitochondria"], box_threshold=0.05)
results_em_2 = filter_large_boxes(results_em_2, em_rgb)
results_em_2 = apply_nms(results_em_2, iou_threshold=0.3)

if len(results_em_2["boxes"]) > 0:
    masks_em_2, iou_em_2 = segment_with_sam2(em_rgb, results_em_2["boxes"])
    show_masks(em_rgb, results_em_2, masks_em_2, iou_em_2,
               title="EM: forced detections at threshold=0.05 fed to SAM 2")
else:
    print("Still no detections even at threshold=0.05.")


### What to observe

This section makes two important points:

**Point 1 -- The pipeline is only as good as its first stage.**
SAM 2 is a prompt-based model. It segments what you point it at. If Grounding DINO
gives it a bad box, SAM 2 will faithfully segment whatever is inside that bad box --
it has no way of knowing the box was wrong. Garbage in, garbage out.

**Point 2 -- SAM 2 handles EM texture better than Grounding DINO.**
Even when the box locations are unreliable, SAM 2's masks may snap to visible boundaries
in the EM image -- dark oval structures, membranes, organelle edges.
SAM 2's image encoder was trained on a much broader visual distribution than
Grounding DINO's text-vision alignment training. SAM 2 sees edges; Grounding DINO sees concepts.

This is the key motivation for domain-specific models, covered in a later notebook.


---
## Section 7: Side-by-Side Comparison -- Boxes vs Masks

How much does the mask actually add over the box?
Here we visualise both for the COCO and kidney images side by side.

This is the visual payoff for the two-notebook arc.


In [ ]:
def show_boxes_on_ax(image, results, title, ax):
    ax.imshow(image)
    ax.set_title(title, fontsize=11, pad=6)
    ax.axis("off")
    boxes  = results["boxes"].cpu().numpy()
    scores = results["scores"].cpu().numpy()
    labels = results.get("text_labels", results.get("labels", []))
    colors = plt.cm.tab10.colors
    for i, (box, score, label) in enumerate(zip(boxes, scores, labels)):
        x0, y0, x1, y1 = box
        color = colors[i % len(colors)]
        rect = patches.Rectangle(
            (x0, y0), x1-x0, y1-y0,
            linewidth=2, edgecolor=color, facecolor="none"
        )
        ax.add_patch(rect)
        ax.text(x0, y0-5, f"{label} {score:.2f}", color="white", fontsize=7,
                bbox=dict(facecolor=color, alpha=0.85, pad=1, edgecolor="none"))


def show_masks_on_ax(image, results, masks, iou_scores, title, ax, alpha=0.45):
    ax.imshow(image)
    ax.set_title(title, fontsize=11, pad=6)
    ax.axis("off")
    boxes  = results["boxes"].cpu().numpy()
    scores = results["scores"].cpu().numpy()
    labels = results.get("text_labels", results.get("labels", []))
    colors = plt.cm.tab10.colors
    for i, (box, score, label) in enumerate(zip(boxes, scores, labels)):
        color = colors[i % len(colors)]
        if i < len(masks):
            mask = masks[i]
            colored = np.zeros((*mask.shape, 4), dtype=float)
            colored[mask] = [*color[:3], alpha]
            ax.imshow(colored)
        x0, y0, x1, y1 = box
        rect = patches.Rectangle(
            (x0, y0), x1-x0, y1-y0,
            linewidth=1.5, edgecolor=color, facecolor="none", linestyle="--"
        )
        ax.add_patch(rect)
        iou = iou_scores[i] if i < len(iou_scores) else 0.0
        ax.text(x0, y0-5, f"{label} iou={iou:.2f}", color="white", fontsize=7,
                bbox=dict(facecolor=color, alpha=0.85, pad=1, edgecolor="none"))


fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# COCO boxes-only (re-run detection without SAM 2)
results_coco_boxes = detect(coco_image, ["a cat", "a remote control"], box_threshold=0.25)
results_coco_boxes = filter_large_boxes(results_coco_boxes, coco_image)
results_coco_boxes = apply_nms(results_coco_boxes)

show_boxes_on_ax(coco_image, results_coco_boxes,
                 "COCO -- boxes only (Notebook 1)", axes[0, 0])
show_masks_on_ax(coco_image, results_coco, masks_coco, iou_coco,
                 "COCO -- pixel masks (Notebook 2)", axes[0, 1])

# Kidney boxes-only
results_kidney_boxes = detect(kidney_image, ["glomerulus"], box_threshold=0.25)
results_kidney_boxes = filter_large_boxes(results_kidney_boxes, kidney_image)
results_kidney_boxes = apply_nms(results_kidney_boxes)

show_boxes_on_ax(kidney_image, results_kidney_boxes,
                 "Kidney -- boxes only (Notebook 1)", axes[1, 0])
show_masks_on_ax(kidney_image, results_kidney, masks_kidney, iou_kidney,
                 "Kidney -- pixel masks (Notebook 2)", axes[1, 1])

plt.suptitle("Boxes vs Masks: what SAM 2 adds to the pipeline", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


---
## Section 8: Summary and What Is Next

### What this pipeline can do today

For natural images and some colour microscopy images (H&E, brightfield), the
Grounding DINO + SAM 2 pipeline produces usable annotation starting points from
a single text prompt. The masks are precise enough to use as initial annotations
that a human then reviews and corrects -- significantly faster than annotating from scratch.

### What it cannot do yet

EM and fluorescence microscopy remain out of reach without adaptation.
The failure is in the detection stage: if Grounding DINO cannot reliably localise
the object, SAM 2 has nothing to work with.

### The path forward

Two directions exist for scientific images:

**Option 1 -- Fine-tune Grounding DINO** on a small set of your own annotated images.
Even 50-100 annotated examples can dramatically improve detection on a specific domain.

**Option 2 -- Bypass detection, use SAM 2 directly with point prompts.**
If you can click on a mitochondrion, SAM 2 will segment it precisely even in EM,
because its image encoder handles low-level boundaries well.
This is the human-in-the-loop annotation workflow.

Both directions are covered in later notebooks in this series.

---


